# MAP Finalized Single-Model Pipeline

This notebook turns your original `final.ipynb` into a practical Kaggle-ready workflow:

- keeps the writeup's core preprocessing idea (`is_correct` + structured prompt)
- fixes validation leakage from singleton duplication
- evaluates both raw and family-filtered MAP@3
- runs a final full-data fit and exports artifacts for a separate Kaggle submission notebook

This is **not** a full reproduction of the linked 45th-place 6-model ensemble. It is a cleaned-up, realistic single-model version that still borrows the most useful ensemble insight: **family-based filtering**.


In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets scikit-learn joblib


In [ ]:
import json
import os
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training
from scipy.special import softmax
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
# =========================
# CONFIG
# =========================
DEBUG = False
SEED = 42
VAL_SIZE = 0.10
MAX_LENGTH = 256
MODEL_NAME = "Qwen/Qwen3-4B"
USE_4BIT = True
RUN_FINAL_FULL_TRAIN = True
EXPORT_MERGED_MODEL = True

TRAIN_PATH = "/kaggle/input/map-charting-student-math-misunderstandings/train.csv"
TEST_PATH = "/kaggle/input/map-charting-student-math-misunderstandings/test.csv"

OUTPUT_DIR = Path("/kaggle/working/map_qwen3_single_model")
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
MERGED_MODEL_DIR = ARTIFACT_DIR / "merged_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 1 if DEBUG else 3
FINAL_FULL_TRAIN_EPOCHS = EPOCHS
TRAIN_BATCH_SIZE = 2 if USE_4BIT else 4
EVAL_BATCH_SIZE = 8
GRAD_ACCUM = 4 if DEBUG else 8
LEARNING_RATE = 2e-4

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


In [ ]:
# =========================
# HELPERS
# =========================
def build_correct_answer_map(train_df: pd.DataFrame) -> dict:
    idx = train_df["Category"].astype(str).str.split("_").str[0] == "True"
    correct = train_df.loc[idx].copy()
    correct["count_per_pair"] = correct.groupby(["QuestionId", "MC_Answer"])["MC_Answer"].transform("count")
    correct = correct.sort_values("count_per_pair", ascending=False).drop_duplicates(["QuestionId"])
    return {
        int(qid): str(ans)
        for qid, ans in zip(correct["QuestionId"], correct["MC_Answer"])
    }

def add_correctness_features(df: pd.DataFrame, correct_by_q: dict) -> pd.DataFrame:
    df = df.copy()
    df["QuestionText"] = df["QuestionText"].fillna("")
    df["StudentExplanation"] = df["StudentExplanation"].fillna("")
    df["MC_Answer"] = df["MC_Answer"].fillna("")

    def get_correct_answer(qid):
        return correct_by_q.get(int(qid)) if pd.notna(qid) else None

    df["correct_answer"] = df["QuestionId"].map(get_correct_answer)
    df["is_correct"] = np.where(
        df["correct_answer"].notna(),
        (df["MC_Answer"] == df["correct_answer"]).astype(int),
        pd.NA,
    )
    df["family_prefix"] = df["is_correct"].map(
        lambda x: None if pd.isna(x) else ("True_" if int(x) == 1 else "False_")
    )
    return df

def format_input(row: pd.Series) -> str:
    if pd.isna(row["is_correct"]):
        correct_text = "Unknown"
    else:
        correct_text = "Yes" if int(row["is_correct"]) == 1 else "No"
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"Is Correct Answer: {correct_text}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )

def map3_from_top3(pred_top3, true_labels):
    total = 0.0
    for pred, truth in zip(pred_top3, true_labels):
        for rank, label in enumerate(pred[:3], start=1):
            if label == truth:
                total += 1.0 / rank
                break
    return total / max(len(true_labels), 1)

def build_family_fillers(classes, train_df):
    fillers = {}
    target_counts = train_df["target"].value_counts()
    for prefix in ["True_", "False_"]:
        family_classes = [c for c in classes if c.startswith(prefix)]
        family_classes = sorted(
            family_classes,
            key=lambda c: (-target_counts.get(c, 0), c),
        )
        if prefix == "True_":
            preferred = [c for c in ["True_Correct:NA", "True_Neither:NA"] if c in family_classes]
        else:
            preferred = [c for c in ["False_Neither:NA"] if c in family_classes]
        seen = set(preferred)
        fillers[prefix] = preferred + [c for c in family_classes if c not in seen]
    return fillers

def select_top3(prob_row, classes, family_prefix=None, family_fillers=None):
    ranked = sorted(zip(classes, prob_row), key=lambda x: x[1], reverse=True)
    if family_prefix:
        ranked = [x for x in ranked if x[0].startswith(family_prefix)]
    selected = [label for label, _ in ranked[:3]]
    if family_prefix and family_fillers:
        for filler in family_fillers.get(family_prefix, []):
            if filler not in selected:
                selected.append(filler)
            if len(selected) == 3:
                break
    if len(selected) < 3:
        for filler, _ in sorted(zip(classes, prob_row), key=lambda x: x[1], reverse=True):
            if filler not in selected:
                selected.append(filler)
            if len(selected) == 3:
                break
    return selected[:3]

def evaluate_with_family_filter(logits, labels, family_prefixes, classes, family_fillers):
    probs = softmax(logits, axis=1)
    raw_top3 = [select_top3(row, classes) for row in probs]
    filtered_top3 = [
        select_top3(row, classes, family_prefix=family_prefixes[i], family_fillers=family_fillers)
        for i, row in enumerate(probs)
    ]
    true_labels = [classes[i] for i in labels]
    return {
        "map3_raw": map3_from_top3(raw_top3, true_labels),
        "map3_family": map3_from_top3(filtered_top3, true_labels),
        "raw_top3": raw_top3,
        "filtered_top3": filtered_top3,
    }

def metric_map3(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=1)
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = 0.0
    for i in range(len(labels)):
        for rank, pred_idx in enumerate(top3[i], start=1):
            if pred_idx == labels[i]:
                score += 1.0 / rank
                break
    return {"map3": score / max(len(labels), 1)}


In [ ]:
# =========================
# LOAD DATA
# =========================
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train["Misconception"] = train["Misconception"].fillna("NA")
train["target"] = train["Category"] + ":" + train["Misconception"]

le = LabelEncoder()
train["label"] = le.fit_transform(train["target"])
classes = list(le.classes_)

correct_by_q = build_correct_answer_map(train)
train = add_correctness_features(train, correct_by_q)
test = add_correctness_features(test, correct_by_q)

train["text"] = train.apply(format_input, axis=1)
test["text"] = test.apply(format_input, axis=1)

if DEBUG:
    train = train.sample(n=min(4000, len(train)), random_state=SEED).reset_index(drop=True)

family_fillers = build_family_fillers(classes, train)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Classes:", len(classes))
print("Sample prompt:")
print(train.loc[0, "text"][:700])


In [ ]:
# =========================
# LEAK-SAFE VALIDATION SPLIT
# =========================
# The original notebook duplicated singleton rows BEFORE splitting.
# That leaks identical samples into both train and validation.
# Here, singleton labels stay in train only.

counts = train["label"].value_counts()
singleton_labels = set(counts[counts == 1].index.tolist())

singleton_df = train[train["label"].isin(singleton_labels)].copy()
multi_df = train[~train["label"].isin(singleton_labels)].copy()

train_main, val_df = train_test_split(
    multi_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=multi_df["label"],
)

train_df = (
    pd.concat([train_main, singleton_df], ignore_index=True)
    .sample(frac=1.0, random_state=SEED)
    .reset_index(drop=True)
)
val_df = val_df.reset_index(drop=True)

print("Singleton labels kept in train only:", len(singleton_labels))
print("Train rows:", len(train_df))
print("Val rows:", len(val_df))


In [ ]:
# =========================
# TOKENIZER + MODEL
# =========================
def create_tokenizer(model_name: str):
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tok.pad_token is None:
        tok.add_special_tokens({"pad_token": "[PAD]"})
    return tok

def create_peft_model(tokenizer, num_labels: int):
    use_bf16_local = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if use_bf16_local else (
        torch.float16 if torch.cuda.is_available() else torch.float32
    )
    quant_config = None
    device_map = None

    if USE_4BIT and torch.cuda.is_available():
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
        )
        device_map = "auto"

    mdl = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
        trust_remote_code=True,
        quantization_config=quant_config,
        device_map=device_map,
        torch_dtype=compute_dtype,
    )
    mdl.config.pad_token_id = tokenizer.pad_token_id
    mdl.config.use_cache = False
    mdl.resize_token_embeddings(len(tokenizer))

    if quant_config is not None:
        mdl = prepare_model_for_kbit_training(mdl)

    lora_config = LoraConfig(
        r=64,
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        modules_to_save=["score"],
    )

    mdl = get_peft_model(mdl, lora_config)
    return mdl, use_bf16_local

tokenizer = create_tokenizer(MODEL_NAME)
model, use_bf16 = create_peft_model(tokenizer, len(classes))
model.print_trainable_parameters()


In [ ]:
# =========================
# DATASETS
# =========================
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False).map(
    tokenize_batch, batched=True, remove_columns=["text"]
)
val_ds = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False).map(
    tokenize_batch, batched=True, remove_columns=["text"]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
# =========================
# TRAIN
# =========================
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "trainer_output"),
    overwrite_output_dir=True,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=25 if DEBUG else 100,
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=metric_map3,
)

trainer.train()


In [ ]:
# =========================
# VALIDATION CHECK
# =========================
val_output = trainer.predict(val_ds)
val_summary = evaluate_with_family_filter(
    logits=val_output.predictions,
    labels=val_df["label"].to_numpy(),
    family_prefixes=val_df["family_prefix"].tolist(),
    classes=classes,
    family_fillers=family_fillers,
)

print("Validation MAP@3 (raw):", round(val_summary["map3_raw"], 5))
print("Validation MAP@3 (family filtered):", round(val_summary["map3_family"], 5))
print("Example filtered predictions:")
for i in range(min(3, len(val_df))):
    print("- truth:", val_df.loc[i, "target"])
    print("  pred :", val_summary["filtered_top3"][i])


In [ ]:
# =========================
# FINAL FULL-DATA TRAIN FOR EXPORT
# =========================
export_trainer = trainer
export_note = "best validation model"

if RUN_FINAL_FULL_TRAIN:
    print("Starting final full-data training pass for export...")
    full_train_df = train.copy().reset_index(drop=True)
    full_train_ds = Dataset.from_pandas(full_train_df[["text", "label"]], preserve_index=False).map(
        tokenize_batch, batched=True, remove_columns=["text"]
    )

    del trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    final_model, final_use_bf16 = create_peft_model(tokenizer, len(classes))
    use_bf16 = final_use_bf16
    final_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / "final_full_train"),
        overwrite_output_dir=True,
        num_train_epochs=FINAL_FULL_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        eval_strategy="no",
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=25 if DEBUG else 100,
        bf16=final_use_bf16,
        fp16=torch.cuda.is_available() and not final_use_bf16,
        gradient_checkpointing=True,
        report_to="none",
        remove_unused_columns=False,
    )

    export_trainer = Trainer(
        model=final_model,
        args=final_args,
        train_dataset=full_train_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    export_trainer.train()
    export_note = "final full-data model"

print("Export source:", export_note)


In [ ]:
# =========================
# SAVE ARTIFACTS
# =========================
export_trainer.model.save_pretrained(ARTIFACT_DIR)
tokenizer.save_pretrained(ARTIFACT_DIR)
joblib.dump(le, ARTIFACT_DIR / "label_encoder.joblib")

metadata = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "num_labels": len(classes),
    "family_fillers": family_fillers,
    "correct_answer_map": {str(k): v for k, v in correct_by_q.items()},
}
with open(ARTIFACT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

if EXPORT_MERGED_MODEL:
    print("Exporting merged offline model for the submission notebook...")
    del val_output, val_summary
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    export_dtype = torch.bfloat16 if use_bf16 else (torch.float16 if torch.cuda.is_available() else torch.float32)
    export_base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(classes),
        trust_remote_code=True,
        torch_dtype=export_dtype,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    export_base.config.pad_token_id = tokenizer.pad_token_id
    export_base.resize_token_embeddings(len(tokenizer))

    merged_model = PeftModel.from_pretrained(export_base, ARTIFACT_DIR).merge_and_unload()
    merged_model.save_pretrained(
        MERGED_MODEL_DIR,
        safe_serialization=True,
        max_shard_size="2GB",
    )
    tokenizer.save_pretrained(MERGED_MODEL_DIR)

print("Saved artifacts to:", ARTIFACT_DIR)
print("Files:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", path.name)
